In [2]:
import pandas as pd
df = pd.read_csv("/content/healthcare_data_cleaning_dataset.csv")
df.head()

,Patient_ID,Age,Gender,City,Diagnosis,Hospital_Visits,Treatment_Cost,Insurance_Coverage,Admission_Date
0,17270,35.0,Male,Bangalore,Hypertension,13,41010.0,1,2023-11-30
1,10860,21.0,Female,Hyderabad,Flu,11,12194.0,1,2023-02-23
2,15390,77.0,Female,Bangalore,Asthma,2,45086.0,0,2023-03-14
3,15191,79.0,Female,Mumbai,Asthma,13,40842.0,0,2023-08-01
4,15734,60.0,Female,Delhi,Asthma,1,9873.0,1,2023-06-20


In [16]:
# Q1. Missing Data Identification
# Scenario: The hospital suspects incomplete patient records.
# Identify missing values in each column

missing_values = df.isnull().sum()

#Calculate percentage of missing data

missing_percentage = (df.isnull().sum() / len(df)) * 100

#combine results

missing_data = pd.DataFrame({
    'Missing_Values': missing_values,
    'Percentage': missing_percentage
})

print(missing_data)

                    Missing_Values  Percentage
Patient_ID                       0         0.0
Age                              0         0.0
Gender                           0         0.0
City                             0         0.0
Diagnosis                        0         0.0
Hospital_Visits                  0         0.0
Treatment_Cost                   0         0.0
Insurance_Coverage               0         0.0
Admission_Date                   0         0.0
Log_Treatment_Cost               0         0.0


In [17]:
# Q2. Handling Missing Age
# Scenario:  Age is critical for medical analysis, but some values are missing.
print(df['Age'].describe())
# Replace missing Age values with an appropriate method

df['Age'].fillna(df['Age'].median(), inplace=True)
print(df['Age'].isnull().sum())


count    5001.000000
mean       49.637872
std        27.190164
min         0.000000
25%        27.000000
50%        50.000000
75%        72.000000
max        99.000000
Name: Age, dtype: float64
0


/tmp/ipykernel_2923/1539579323.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)


Why Median?
Age data may contain outliers
Median is robust and less affected by extreme values
Better than mean for skewed distributions

In [18]:
# Q3. Handling Missing Treatment Cost
# Scenario: Treatment cost is highly skewed due to expensive treatments.
# Check skewness
print(df['Treatment_Cost'].skew())

# Handle missing Treatment_Cost values
df['Treatment_Cost'].fillna(df['Treatment_Cost'].median(), inplace=True)

# Verify
print(df['Treatment_Cost'].isnull().sum())

4.514908884751754
0


/tmp/ipykernel_2923/2327365110.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Treatment_Cost'].fillna(df['Treatment_Cost'].median(), inplace=True)


Why Median?
Treatment cost is highly skewed
Expensive treatments create extreme values
Median gives a better central value

In [19]:
# Q.4 - Duplicate Patient Records
# Scenario: Some patient records were entered multiple times.
# Dataset size before removing duplicates
before = df.shape[0]

# Identify duplicate rows
duplicates = df[df.duplicated()]
print("Duplicate Rows:")
print(duplicates)

# Remove duplicates
df = df.drop_duplicates()

# Compare dataset size before and after

# Dataset size after removal
after = df.shape[0]

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Duplicate Rows:
Empty DataFrame
Columns: [Patient_ID, Age, Gender, City, Diagnosis, Hospital_Visits, Treatment_Cost, Insurance_Coverage, Admission_Date, Log_Treatment_Cost]
Index: []
Rows before: 5001
Rows after: 5001
Duplicates removed: 0


In [20]:
# Q5. Invalid Age Values (Data Quality Check)
# Scenario: Some patients have unrealistic age values (e.g., >100 or <0).
# Some patients have unrealistic age values (e.g., >100 or <0).
# Detect such records
invalid_age = df[(df['Age'] < 0) | (df['Age'] > 100)]

print(invalid_age)

# Decide whether to remove or correct them
df = df[(df['Age'] >= 0) & (df['Age'] <= 100)]

print("Updated dataset shape:", df.shape)



Empty DataFrame
Columns: [Patient_ID, Age, Gender, City, Diagnosis, Hospital_Visits, Treatment_Cost, Insurance_Coverage, Admission_Date, Log_Treatment_Cost]
Index: []
Updated dataset shape: (5001, 10)


Why?
Negative age or age >100 is unrealistic
Such records reduce data quality

In [21]:
# Q6. Outlier Detection (Treatment Cost)
# Scenario: Extreme treatment costs are affecting analysis.
# Detect outliers using IQR method
# Calculate Q1 and Q3
Q1 = df['Treatment_Cost'].quantile(0.25)
Q3 = df['Treatment_Cost'].quantile(0.75)

# IQR
IQR = Q3 - Q1

# Lower and Upper limits
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Detect outliers
outliers = df[(df['Treatment_Cost'] < lower_limit) |
              (df['Treatment_Cost'] > upper_limit)]

# Display number of outliers

print("Number of Outliers:", outliers.shape[0])

print(outliers)

Number of Outliers: 59
      Patient_ID   Age  Gender       City     Diagnosis  Hospital_Visits  \
4617       10896  88.0  Female    Chennai      COVID-19                3   
2642       19262  50.0  Female  Hyderabad        Asthma                9   
4292       12430  25.0    Male    Chennai      Diabetes               11   
1650       12356  50.0  Female      Delhi      COVID-19                4   
183        15855  36.0    Male     Mumbai      Diabetes                1   
2561       13525  50.0    Male  Bangalore      COVID-19               19   
53         13152  91.0  Female    Chennai      COVID-19               13   
4780       10717  52.0  Female     Mumbai      Diabetes               17   
90         12027  50.0  Female     Mumbai           Flu                1   
2166       10593  44.0  Female      Delhi      COVID-19               18   
86         10064  28.0  Female  Hyderabad  Hypertension               16   
4060       15944  72.0  Female     Mumbai        Asthma          

In [22]:
# Q7. Outlier Treatment
# Scenario:The business team wants to retain all records.

# Apply capping (Winsorization) on Treatment_Cost

# Calculate 5th and 95th percentile
lower_cap = df['Treatment_Cost'].quantile(0.05)
upper_cap = df['Treatment_Cost'].quantile(0.95)

# Apply capping
df['Treatment_Cost'] = df['Treatment_Cost'].clip(lower=lower_cap,
                                                 upper=upper_cap)

print(df['Treatment_Cost'].describe())

count     5001.000000
mean     25146.673265
std      14334.381299
min       2916.000000
25%      12426.000000
50%      24644.000000
75%      37706.000000
max      48236.000000
Name: Treatment_Cost, dtype: float64


In [23]:
# Q8. Transformation
# Scenario: Treatment cost is highly skewed.
# Apply log transformation
# Create a new column
# Compare before vs after distribution

import numpy as np

# Create log transformed column
df['Log_Treatment_Cost'] = np.log(df['Treatment_Cost'])

# Compare distributions
print(df[['Treatment_Cost', 'Log_Treatment_Cost']].head())


      Treatment_Cost  Log_Treatment_Cost
2932         32149.0           10.378137
4617         48236.0           10.783861
2642         48236.0           10.783861
4837         46469.0           10.746541
2702         12398.0            9.425290


In [24]:
# Q9. Time-Based Missing Handling
# Scenario: Admission dates should follow a logical sequence.
# Convert to datetime
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])

# Sort data by Admission_Date
df = df.sort_values(by='Admission_Date')

# Apply forward fill or backward fill where appropriate
# Forward fill
df.fillna(method='ffill', inplace=True)

print(df.head())

      Patient_ID   Age  Gender     City     Diagnosis  Hospital_Visits  \
2932       19591  91.0    Male  Chennai        Asthma                3   
4329       17692  64.0    Male  Chennai  Hypertension               19   
866        15966  49.0  Female   Mumbai  Hypertension               10   
1650       12356  50.0  Female    Delhi      COVID-19                4   
4292       12430  25.0    Male  Chennai      Diabetes               11   

      Treatment_Cost  Insurance_Coverage Admission_Date  Log_Treatment_Cost  
2932         32149.0                   0     2023-01-01           10.378137  
4329         36377.0                   1     2023-01-01           10.501692  
866          43720.0                   1     2023-01-01           10.685561  
1650         48236.0                   1     2023-01-01           10.783861  
4292         48236.0                   0     2023-01-01           10.783861  


/tmp/ipykernel_2923/2818272059.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


Why Forward Fill?
Medical records usually follow time sequence
Previous valid record can logically continue forward
Useful in time-series healthcare data